# Figure S24 — Amphibole chemometry

## 1. Imports and shared 00_Setup style

This notebook uses the current S1 supplemental workbook, Arial typography, 8-point ordinary text, 9-point bold uppercase panel labels, full plot boxes, light gridlines, and editable PDF/SVG text. The code is linear and contains no custom functions.

In [1]:
from pathlib import Path
import hashlib
import inspect
import re

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
import numpy as np
import pandas as pd
from packaging.version import Version
from PIL import Image
import Thermobar as pt

SINGLE_COLUMN_WIDTH_IN = 3.55
DOUBLE_COLUMN_WIDTH_IN = 7.25
MAXIMUM_FIGURE_HEIGHT_IN = 7.8
FIGURE_WIDTH_IN = DOUBLE_COLUMN_WIDTH_IN
FIGURE_HEIGHT_IN = MAXIMUM_FIGURE_HEIGHT_IN
BASE_TEXT_SIZE_PT = 8.0
PANEL_LABEL_SIZE_PT = 9.0
MINIMUM_LINE_WIDTH_PT = 0.28
EXPORT_DPI = 300
PUBLICATION_FONT = "Arial"

installed_font_names = {entry.name for entry in font_manager.fontManager.ttflist}
if PUBLICATION_FONT not in installed_font_names:
    raise RuntimeError("Arial is required by 00_Setup.")

mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": [PUBLICATION_FONT],
    "font.size": BASE_TEXT_SIZE_PT,
    "axes.labelsize": BASE_TEXT_SIZE_PT,
    "axes.titlesize": BASE_TEXT_SIZE_PT,
    "xtick.labelsize": BASE_TEXT_SIZE_PT,
    "ytick.labelsize": BASE_TEXT_SIZE_PT,
    "legend.fontsize": 6.0,
    "legend.title_fontsize": BASE_TEXT_SIZE_PT,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.spines.top": True,
    "axes.spines.right": True,
    "axes.linewidth": 0.5,
    "lines.linewidth": 0.75,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "xtick.major.width": 0.5,
    "ytick.major.width": 0.5,
    "savefig.dpi": EXPORT_DPI,
    "savefig.facecolor": "white",
    "savefig.transparent": False,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
})


## 2. Figure paths and shared sample configuration

S23 uses the S1 workbook only. No legacy or external data files are read.

In [2]:
PROJECT_ROOT_OVERRIDE = None
PROJECT_ROOT = PROJECT_ROOT_OVERRIDE
if PROJECT_ROOT is None:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "SA_MI_Draft_FinalFigures_Work" / "DataSheets").is_dir():
            PROJECT_ROOT = candidate
            break
if PROJECT_ROOT is None:
    raise FileNotFoundError("Run this notebook from the project or set PROJECT_ROOT_OVERRIDE.")

PROJECT_ROOT = Path(PROJECT_ROOT).expanduser().resolve()
WORK_ROOT = PROJECT_ROOT / "SA_MI_Draft_FinalFigures_Work"
FIGURE_DIRECTORY = WORK_ROOT / "Nov6_AllFigures" / "Figure_S24_Amph_Chemometry"
OUTPUT_DIRECTORY = FIGURE_DIRECTORY / "Figure_S24_Amph_Chemometry_Final"
OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)

S1_PATH = WORK_ROOT / "DataSheets" / "Updated_Oct20" / "Dechert_S1_AllData.xlsx"
if not S1_PATH.is_file():
    raise FileNotFoundError(f"Missing S1 workbook: {S1_PATH}")
SOURCE_HASH_BEFORE = hashlib.sha256(S1_PATH.read_bytes()).hexdigest()

SAMPLE_ORDER = ["74D", "74A", "75", "76A"]
SAMPLE_COLORS = {"74D": "#F4A261", "74A": "#D55E00", "75": "#56B4E9", "76A": "#0072B2"}
SAMPLE_LABELS = {"74D": "B Middle", "74A": "B Base", "75": "G Top", "76A": "G Middle"}
MINGLED_COLORS = {
    "B Mingled Dark": "#808080",
    "B Mingled Light": "#EE3377",
    "G Mingled Dark": "#000000",
    "G Mingled Light": "#66C2C2",
}
GROUP_ORDER = [
    "High-Al2O3 transects", "B Transect Points", "G Transect Points",
    "B Mingled Dark", "B Mingled Light", "G Mingled Dark", "G Mingled Light",
    "Observed MI / Matrix Glass",
]


## 3. Read and prepare the established S1 populations

In [3]:
transect_source = pd.read_excel(S1_PATH, sheet_name="Transects")
mingled_source = pd.read_excel(S1_PATH, sheet_name="Mingled_Minerals")
mi_source = pd.read_excel(S1_PATH, sheet_name="MI_Hosts_Plag_Pxy")

all_linescans = transect_source.copy().rename(columns={"Sample": "Sample_Group"})
scan_counts = all_linescans.groupby("Linescan_ID")["Predict_Mineral"].value_counts(dropna=False)
scan_total = all_linescans.groupby("Linescan_ID").size()
dominant = scan_counts.groupby(level=0).idxmax().map(lambda value: value[1])
all_linescans["Dominant_Mineral"] = all_linescans["Linescan_ID"].map(dominant)
dominant_fraction = scan_counts.groupby(level=0).max().div(scan_total)
all_linescans["Dominant_Mineral_Fraction"] = all_linescans["Linescan_ID"].map(dominant_fraction)
assert all_linescans["Sample_Group"].isin(SAMPLE_ORDER).all()

RAW_OXIDE_MAP = {
    "SiO2": "SiO2_SEMMineral_Raw", "TiO2": "TiO2_SEMMineral_Raw",
    "Al2O3": "Al2O3_SEMMineral_Raw", "FeOt": "FeOt_SEMMineral_Raw",
    "MnO": "MnO_SEMMineral_Raw", "MgO": "MgO_SEMMineral_Raw",
    "CaO": "CaO_SEMMineral_Raw", "Na2O": "Na2O_SEMMineral_Raw",
    "K2O": "K2O_SEMMineral_Raw",
}
MINGLED_RAW_OXIDE_MAP = {
    "SiO2": "SiO2_SEMMineral_Raw", "TiO2": "TiO2_SEMMineral_Raw",
    "Al2O3": "Al2O3_SEMMineral_Raw", "FeOt": "FeOt_SEMMineral_Raw",
    "MnO": "MnO_SEMMineral_Raw", "MgO": "MgO_SEMMineral_Raw",
    "CaO": "CaO_SEMMineral_Raw", "Na2O": "Na2O_SEMMineral_Raw",
    "K2O": "K2O_SEMMineral_Raw", "Cr2O3": "Cr2O3_SEMMineral_Raw",
}

linescan_score = pd.to_numeric(all_linescans["Prediction_Score"], errors="coerce")
transect_mask = (
    all_linescans["Dominant_Mineral"].astype("string").str.casefold().eq("amphibole")
    & all_linescans["Dominant_Mineral_Fraction"].ge(0.95)
    & all_linescans["Predict_Mineral"].astype("string").str.casefold().eq("amphibole")
    & linescan_score.gt(0.85)
    & pd.to_numeric(all_linescans["Distance (µm)"], errors="coerce").notna()
)
transects = all_linescans.loc[transect_mask].copy()
transects["Source Data Row"] = transects.index
for oxide, source_column in RAW_OXIDE_MAP.items():
    transects[f"{oxide}_Thermobar"] = pd.to_numeric(transects[source_column], errors="coerce")
transects["Cr2O3_Thermobar"] = 0.0
transects["Cl_Thermobar"] = pd.to_numeric(transects["Cl Ox%"], errors="coerce").fillna(0.0) if "Cl Ox%" in transects.columns else 0.0
transects = transects.dropna(subset=[f"{oxide}_Thermobar" for oxide in RAW_OXIDE_MAP]).reset_index(drop=True)
transects["Analysis ID"] = [
    f"transect__{line_id}__datasheetrow_{int(row)}"
    for line_id, row in zip(transects["Linescan_ID"], transects["Source Data Row"])
]
HIGH_AL_LINES = {
    "74A2_amph12_linescan_lr",
    "74A2_amph7_linescan_rl",
    "74A2_px14_linescan_lr",
}
transects["Group"] = np.select(
    [
        transects["Linescan_ID"].isin(HIGH_AL_LINES),
        transects["Sample_Group"].isin(["74D", "74A"]),
        transects["Sample_Group"].isin(["75", "76A"]),
    ],
    GROUP_ORDER[:3],
    default="Unassigned",
)
if transects["Group"].eq("Unassigned").any():
    raise ValueError("A retained transect could not be assigned to a B/G group.")
transects["Color"] = transects["Sample_Group"].map(SAMPLE_COLORS)
transects.loc[transects["Group"].eq(GROUP_ORDER[0]), "Color"] = "#39FF14"

mingled_minerals = mingled_source.copy().rename(columns={"Type of Pumice": "Mafic/Felsic"})
mingled_score = pd.to_numeric(mingled_minerals["Prediction_Score"], errors="coerce")
mingled_mask = (
    mingled_minerals["Predict_Mineral"].astype("string").str.casefold().eq("amphibole")
    & mingled_score.gt(0.85)
    & mingled_minerals["Dataset"].isin(["74Ming", "76Ming"])
)
mingled = mingled_minerals.loc[mingled_mask].copy()
mingled["Source Data Row"] = mingled.index
for oxide, source_column in MINGLED_RAW_OXIDE_MAP.items():
    mingled[f"{oxide}_Thermobar"] = pd.to_numeric(mingled[source_column], errors="coerce")
mingled = mingled.dropna(subset=[f"{oxide}_Thermobar" for oxide in MINGLED_RAW_OXIDE_MAP]).reset_index(drop=True)
mingled[list(MINGLED_RAW_OXIDE_MAP.values())] = mingled[list(MINGLED_RAW_OXIDE_MAP.values())].clip(lower=0)
texture = mingled["Mafic/Felsic"].astype("string").str.strip().str.casefold()
mingled["Group"] = np.select(
    [
        mingled["Dataset"].eq("74Ming") & (texture.str.startswith("m") | texture.eq("dark")),
        mingled["Dataset"].eq("74Ming") & (texture.str.startswith("f") | texture.eq("light")),
        mingled["Dataset"].eq("76Ming") & (texture.str.startswith("m") | texture.eq("dark")),
        mingled["Dataset"].eq("76Ming") & (texture.str.startswith("f") | texture.eq("light")),
    ],
    GROUP_ORDER[3:7],
    default="Unassigned",
)
if mingled["Group"].eq("Unassigned").any():
    raise ValueError("A retained mingled amphibole could not be assigned to a B/G group.")
mingled["Analysis ID"] = [
    f"mingled__{dataset}__{name}__datasheetrow_{int(row)}"
    for dataset, name, row in zip(mingled["Dataset"], mingled["Sample Name"], mingled["Source Data Row"])
]
mingled["Color"] = mingled["Group"].map(MINGLED_COLORS)

sem_discard = mi_source["Discard_SEMGlass_Raw"].astype("string").str.strip().str.casefold().str.startswith("yes", na=False)
glass = mi_source.loc[~sem_discard, ["base_name", "mineral_phase", "Discard_SIMS", "SiO2_SEMGlass_AnhydoNorm"]].copy()
glass["Sample key"] = glass["base_name"].astype("string").str.extract(r"^(74D|74A|75|76A)", expand=False)
glass["SiO2 observed"] = pd.to_numeric(glass["SiO2_SEMGlass_AnhydoNorm"], errors="coerce")
glass["SIMS discard"] = glass["Discard_SIMS"].astype("string").str.strip().str.casefold().eq("yes").fillna(False)
glass["Is matrix glass"] = glass["base_name"].astype("string").str.contains("_MTX", case=False, na=False)
glass["Is melt inclusion"] = glass["base_name"].astype("string").str.contains("_MI", case=False, na=False) & ~glass["Is matrix glass"]
observed_glass = glass.loc[
    glass["Sample key"].isin(SAMPLE_ORDER)
    & glass["SiO2 observed"].notna()
    & (glass["Is matrix glass"] | (glass["Is melt inclusion"] & ~glass["SIMS discard"]))
].copy()
observed_glass["Observed type"] = np.where(observed_glass["Is matrix glass"], "Matrix glass", "Melt inclusion")
observed_glass["Color"] = observed_glass["Sample key"].map(SAMPLE_COLORS)
assert set(transects.loc[transects["Group"].eq(GROUP_ORDER[0]), "Linescan_ID"]) == HIGH_AL_LINES
assert set(mingled["Group"]) == set(GROUP_ORDER[3:7])


## 4. Calculate the five equilibrium-melt SiO2 chemometers

Thermobar remains the calculation engine. The inline pandas-3 compatibility patch is required by the installed Thermobar version; no custom helper functions are defined.

In [4]:
if Version(pd.__version__) >= Version("3.0"):
    import Thermobar.core as tb_core
    import Thermobar.amphibole as tb_amphibole
    for function_name in ("calculate_sites_ridolfi", "get_amp_sites_leake", "get_amp_sites_avferric_zhang"):
        original = getattr(tb_core, function_name)
        try:
            source = inspect.getsource(original)
        except OSError:
            continue
        float_source = re.sub(r"(norm_cations\[['\"][^'\"]+['\"]\]\s*)=0(?![.\d])", r"\1=0.0", source)
        if float_source == source:
            raise RuntimeError(f"Thermobar compatibility patch found no integer initializer in {function_name}.")
        exec(float_source, tb_core.__dict__)
    tb_amphibole.calculate_sites_ridolfi = tb_core.calculate_sites_ridolfi
    tb_amphibole.get_amp_sites_avferric_zhang = tb_core.get_amp_sites_avferric_zhang

transect_comps = pd.DataFrame({f"{oxide}_Amp": transects[f"{oxide}_Thermobar"].astype(float) for oxide in RAW_OXIDE_MAP})
transect_comps["Cr2O3_Amp"] = transects["Cr2O3_Thermobar"].astype(float)
transect_comps["F_Amp"] = 0.0
transect_comps["Cl_Amp"] = transects["Cl_Thermobar"].astype(float)
transect_comps["Sample_ID_Amp"] = transects["Analysis ID"].values

mingled_comps = pd.DataFrame({f"{oxide}_Amp": mingled[f"{oxide}_Thermobar"].astype(float) for oxide in MINGLED_RAW_OXIDE_MAP})
mingled_comps["F_Amp"] = 0.0
mingled_comps["Cl_Amp"] = 0.0
mingled_comps["Sample_ID_Amp"] = mingled["Analysis ID"].values

SIO2_EQUATIONS = {
    "Zhang17 Eq1": {"column": "Melt_SiO2_Eq1_Zhang17", "rmse": 3.78, "temperature_dependent": False, "published_range": (39.6, 79.2), "title": "Zhang et al. (2017) Eq. 1"},
    "Zhang17 Eq2": {"column": "Melt_SiO2_Eq2_Zhang17", "rmse": 4.19, "temperature_dependent": False, "published_range": (39.6, 79.2), "title": "Zhang et al. (2017) Eq. 2"},
    "Zhang17 Eq3": {"column": "Melt_SiO2_Eq3_Zhang17", "rmse": 4.37, "temperature_dependent": True, "published_range": (39.6, 79.2), "title": "Zhang et al. (2017) Eq. 3"},
    "Zhang17 Eq4": {"column": "Melt_SiO2_Eq4_Zhang17", "rmse": 4.51, "temperature_dependent": False, "published_range": (39.6, 79.2), "title": "Zhang et al. (2017) Eq. 4"},
    "Putirka16 Eq10": {"column": "Melt_SiO2_Eq10_Put2016", "rmse": 3.60, "temperature_dependent": True, "published_range": None, "title": "Putirka (2016) Eq. 10"},
}

transect_PT = pt.calculate_amp_only_press_temp(amp_comps=transect_comps, equationP="P_Ridolfi2021", equationT="T_Ridolfi2012", Ridolfi_Filter=False).reset_index(drop=True)
transect_melt = pt.calculate_amp_only_melt_comps(amp_comps=transect_comps, T=transect_PT["T_K_calc"]).reset_index(drop=True)
transect_diagnostic = pt.calculate_amp_only_press(amp_comps=transect_comps, equationP="P_Ridolfi2021", classification=True, Ridolfi_Filter=False).reset_index(drop=True)
mingled_PT = pt.calculate_amp_only_press_temp(amp_comps=mingled_comps, equationP="P_Ridolfi2021", equationT="T_Ridolfi2012", Ridolfi_Filter=False).reset_index(drop=True)
mingled_melt = pt.calculate_amp_only_melt_comps(amp_comps=mingled_comps, T=mingled_PT["T_K_calc"]).reset_index(drop=True)
mingled_diagnostic = pt.calculate_amp_only_press(amp_comps=mingled_comps, equationP="P_Ridolfi2021", classification=True, Ridolfi_Filter=False).reset_index(drop=True)

transect_results = pd.DataFrame({"Ridolfi_Input_Check": transect_diagnostic["Input_Check"].fillna(False).astype(bool)})
mingled_results = pd.DataFrame({"Ridolfi_Input_Check": mingled_diagnostic["Input_Check"].fillna(False).astype(bool)})
for equation, spec in SIO2_EQUATIONS.items():
    source_column = spec["column"].removeprefix("Melt_")
    if source_column not in transect_melt or source_column not in mingled_melt:
        raise KeyError(f"Thermobar output missing {source_column}.")
    transect_results[spec["column"]] = pd.to_numeric(transect_melt[source_column], errors="coerce")
    mingled_results[spec["column"]] = pd.to_numeric(mingled_melt[source_column], errors="coerce")

transect_table = transects[["Analysis ID", "Group", "Color"]].copy().reset_index(drop=True)
mingled_table = mingled[["Analysis ID", "Group", "Color"]].copy().reset_index(drop=True)
for column in transect_results.columns:
    transect_table[column] = transect_results[column].to_numpy()
for column in mingled_results.columns:
    mingled_table[column] = mingled_results[column].to_numpy()
amphibole_wide = pd.concat([transect_table, mingled_table], ignore_index=True)
amphibole_wide["Group"] = pd.Categorical(amphibole_wide["Group"], GROUP_ORDER[:-1], ordered=True)
amphibole_wide["Jitter"] = np.random.default_rng(20260916).uniform(-0.22, 0.22, len(amphibole_wide))

chemometry_long_parts = []
for equation, spec in SIO2_EQUATIONS.items():
    part = amphibole_wide.copy()
    part["Equation"] = equation
    part["Calculated melt SiO2 (wt%)"] = pd.to_numeric(part[spec["column"]], errors="coerce")
    part["Finite result"] = np.isfinite(part["Calculated melt SiO2 (wt%)"])
    part["Passes equation input screen"] = part["Finite result"]
    if spec["temperature_dependent"]:
        part["Passes equation input screen"] &= part["Ridolfi_Input_Check"]
    part["Zhang range extrapolation"] = (
        part["Finite result"] & ~part["Calculated melt SiO2 (wt%)"].between(*spec["published_range"], inclusive="both")
        if spec["published_range"] is not None else False
    )
    part["Diagnostic-only finite"] = part["Finite result"] & ~part["Passes equation input screen"]
    chemometry_long_parts.append(part)
chemometry_long = pd.concat(chemometry_long_parts, ignore_index=True)
chemometry_long["Equation"] = pd.Categorical(chemometry_long["Equation"], list(SIO2_EQUATIONS), ordered=True)


## 5. Plot and export Figure S24

In [5]:
AMPHIBOLE_MARKER = "^"
GROUP_POSITIONS = {group: index for index, group in enumerate(GROUP_ORDER)}
XTICK_LABELS = [
    "High-Al$_2$O$_3$\ntransects", "B Transect\nPoints", "G Transect\nPoints",
    "B Mingled\nDark", "B Mingled\nLight", "G Mingled\nDark", "G Mingled\nLight",
    "Observed MI /\nMatrix Glass",
]

figure, axes = plt.subplots(5, 1, figsize=(FIGURE_WIDTH_IN, FIGURE_HEIGHT_IN), sharex=True)
figure.subplots_adjust(left=0.12, right=0.985, top=0.90, bottom=0.22, hspace=0.34)
observed_jitter = np.random.default_rng(20260916).uniform(-0.065, 0.065, len(observed_glass))
amphibole_artists = []
diagnostic_artists = []
range_artists = []
median_artists = []
observed_artists = []

for panel_index, (axis, (equation, spec)) in enumerate(zip(axes, SIO2_EQUATIONS.items())):
    equation_data = chemometry_long.loc[chemometry_long["Equation"].eq(equation)]
    for group in GROUP_ORDER[:-1]:
        group_data = equation_data.loc[equation_data["Group"].eq(group)]
        passing = group_data.loc[group_data["Passes equation input screen"]]
        diagnostic = group_data.loc[group_data["Diagnostic-only finite"]]
        if not passing.empty:
            amphibole_artists.append(axis.scatter(
                GROUP_POSITIONS[group] + passing["Jitter"], passing["Calculated melt SiO2 (wt%)"],
                s=28.0, marker=AMPHIBOLE_MARKER, facecolors=passing["Color"],
                edgecolors="black", linewidths=MINIMUM_LINE_WIDTH_PT, alpha=0.78, zorder=2,
            ))
        if not diagnostic.empty:
            diagnostic_artists.append(axis.scatter(
                GROUP_POSITIONS[group] + diagnostic["Jitter"], diagnostic["Calculated melt SiO2 (wt%)"],
                s=30.0, marker=AMPHIBOLE_MARKER, facecolors="none",
                edgecolors=diagnostic["Color"], linewidths=0.58, alpha=0.95, zorder=3,
            ))
        range_flagged = group_data.loc[group_data["Zhang range extrapolation"]]
        if not range_flagged.empty:
            range_artists.append(axis.scatter(
                GROUP_POSITIONS[group] + range_flagged["Jitter"], range_flagged["Calculated melt SiO2 (wt%)"],
                s=38.0, marker="|", color="0.18", linewidths=0.75, alpha=0.80, zorder=4,
            ))
        median_values = group_data.loc[group_data["Passes equation input screen"], "Calculated melt SiO2 (wt%)"]
        if median_values.notna().any():
            median_artists.append(axis.hlines(
                median_values.median(), GROUP_POSITIONS[group] - 0.28,
                GROUP_POSITIONS[group] + 0.28, color="black", linewidth=1.45, zorder=6,
            ))

    observed_x_center = GROUP_POSITIONS[GROUP_ORDER[-1]]
    for observed_type, offset in (("Melt inclusion", -0.13), ("Matrix glass", 0.13)):
        type_mask = observed_glass["Observed type"].eq(observed_type)
        for sample_key, color in SAMPLE_COLORS.items():
            selected_mask = type_mask & observed_glass["Sample key"].eq(sample_key)
            selected = observed_glass.loc[selected_mask]
            if selected.empty:
                continue
            marker = "o" if observed_type == "Melt inclusion" else "x"
            observed_artists.append(axis.scatter(
                observed_x_center + offset + observed_jitter[selected_mask.to_numpy()],
                selected["SiO2 observed"], s=28.0 if marker == "o" else 34.0,
                marker=marker, color=color, edgecolors="black" if marker == "o" else color,
                linewidths=MINIMUM_LINE_WIDTH_PT, alpha=0.80, zorder=2.5,
            ))
        median_values = observed_glass.loc[type_mask, "SiO2 observed"]
        if median_values.notna().any():
            median_artists.append(axis.hlines(
                median_values.median(), observed_x_center + offset - 0.105,
                observed_x_center + offset + 0.105, color="black", linewidth=1.45, zorder=6,
            ))

    axis.set_xlim(-0.5, len(GROUP_ORDER) - 0.5)
    axis.set_ylim(60.0, 90.0)
    axis.set_yticks(np.arange(60, 91, 5))
    axis.set_ylabel("")
    axis.set_title(
        f"{chr(65 + panel_index)}) {spec['title']} (RMSE = {spec['rmse']:.2f} wt% SiO$_2$)",
        loc="left", fontsize=PANEL_LABEL_SIZE_PT, fontweight="bold", pad=2.5,
    )
    axis.grid(True, axis="y", linestyle=":", linewidth=0.45, color="0.78", alpha=0.65)
    axis.set_axisbelow(True)
    for spine in axis.spines.values():
        spine.set_visible(True)
        spine.set_color("black")
        spine.set_linewidth(0.55)

axes[-1].set_xticks(range(len(GROUP_ORDER)), XTICK_LABELS, rotation=24, ha="right")
axes[-1].tick_params(axis="x", pad=3.0)
figure.text(0.035, 0.50, "Calculated melt SiO$_2$ (wt%)", rotation=90, ha="center", va="center", fontsize=BASE_TEXT_SIZE_PT)

color_handles = [
    Patch(facecolor="#39FF14", edgecolor="black", linewidth=0.35, label="High-Al$_2$O$_3$ transects"),
    Patch(facecolor=SAMPLE_COLORS["74D"], edgecolor="black", linewidth=0.35, label=SAMPLE_LABELS["74D"]),
    Patch(facecolor=SAMPLE_COLORS["74A"], edgecolor="black", linewidth=0.35, label=SAMPLE_LABELS["74A"]),
    Patch(facecolor=SAMPLE_COLORS["75"], edgecolor="black", linewidth=0.35, label=SAMPLE_LABELS["75"]),
    Patch(facecolor=SAMPLE_COLORS["76A"], edgecolor="black", linewidth=0.35, label=SAMPLE_LABELS["76A"]),
    Patch(facecolor=MINGLED_COLORS["B Mingled Dark"], edgecolor="black", linewidth=0.35, label="B mingled dark"),
    Patch(facecolor=MINGLED_COLORS["B Mingled Light"], edgecolor="black", linewidth=0.35, label="B mingled light"),
    Patch(facecolor=MINGLED_COLORS["G Mingled Dark"], edgecolor="black", linewidth=0.35, label="G mingled dark"),
    Patch(facecolor=MINGLED_COLORS["G Mingled Light"], edgecolor="black", linewidth=0.35, label="G mingled light"),
]
symbol_handles = [
    Line2D([], [], marker="^", linestyle="none", markersize=5.5, markerfacecolor="0.55", markeredgecolor="black", label="Amphibole analysis"),
    Line2D([], [], marker="o", linestyle="none", markersize=5.0, markerfacecolor="0.55", markeredgecolor="black", label="Plagioclase-hosted MI"),
    Line2D([], [], marker="s", linestyle="none", markersize=5.0, markerfacecolor="0.55", markeredgecolor="black", label="Pyroxene-hosted MI"),
    Line2D([], [], marker="x", linestyle="none", markersize=5.0, color="0.35", label="Matrix glass"),
    Line2D([], [], marker="^", linestyle="none", markersize=5.5, markerfacecolor="none", markeredgecolor="0.35", label="Diagnostic only"),
    Line2D([], [], marker="|", linestyle="none", markersize=6.0, markeredgewidth=0.75, color="0.18", label="Zhang range extrapolation"),
    Line2D([], [], color="black", linewidth=1.45, label="Median"),
]
color_legend = figure.legend(
    handles=color_handles, title="Samples and Population", loc="upper center",
    bbox_to_anchor=(0.55, 0.995), ncol=4, frameon=True, fancybox=False,
    framealpha=1.0, edgecolor="black", handletextpad=0.35, columnspacing=0.75,
    labelspacing=0.25, borderpad=0.35,
)
color_legend.get_frame().set_linewidth(0.5)
symbol_legend = figure.legend(
    handles=symbol_handles, title="Data Type and Status", loc="lower center",
    bbox_to_anchor=(0.55, 0.025), ncol=4, frameon=True, fancybox=False,
    framealpha=1.0, edgecolor="black", handletextpad=0.35, columnspacing=0.75,
    labelspacing=0.25, borderpad=0.35,
)
symbol_legend.get_frame().set_linewidth(0.5)

output_stem = OUTPUT_DIRECTORY / "Figure_S24_Amph_Chemometry"
figure.savefig(output_stem.with_suffix(".pdf"), format="pdf", dpi=EXPORT_DPI, facecolor="white", transparent=False)
figure.savefig(output_stem.with_suffix(".png"), format="png", dpi=EXPORT_DPI, facecolor="white", transparent=False)
figure.savefig(output_stem.with_suffix(".svg"), format="svg", dpi=EXPORT_DPI, facecolor="white", transparent=False)
plt.close(figure)


/var/folders/5k/6pnfdpxn5419w3kg2f6c8pwc0000gn/T/ipykernel_40452/30627134.py:58: UserWarning: You passed an edgecolor/edgecolors ('#F4A261') for an unfilled marker ('x').  Matplotlib is ignoring the edgecolor in favor of the facecolor.  This behavior may change in the future.
  observed_artists.append(axis.scatter(
/var/folders/5k/6pnfdpxn5419w3kg2f6c8pwc0000gn/T/ipykernel_40452/30627134.py:58: UserWarning: You passed an edgecolor/edgecolors ('#D55E00') for an unfilled marker ('x').  Matplotlib is ignoring the edgecolor in favor of the facecolor.  This behavior may change in the future.
  observed_artists.append(axis.scatter(
/var/folders/5k/6pnfdpxn5419w3kg2f6c8pwc0000gn/T/ipykernel_40452/30627134.py:58: UserWarning: You passed an edgecolor/edgecolors ('#56B4E9') for an unfilled marker ('x').  Matplotlib is ignoring the edgecolor in favor of the facecolor.  This behavior may change in the future.
  observed_artists.append(axis.scatter(
/var/folders/5k/6pnfdpxn5419w3kg2f6c8pwc0000gn/T/

## 6. Final export checks

In [6]:
assert hashlib.sha256(S1_PATH.read_bytes()).hexdigest() == SOURCE_HASH_BEFORE
assert len(axes) == 5
assert [axis.get_title(loc="left").split()[0] for axis in axes] == ["A)", "B)", "C)", "D)", "E)"]
assert all(axis.get_ylim() == (60.0, 90.0) for axis in axes)
assert len(amphibole_artists) > 0
assert len(median_artists) > 0
for suffix in (".pdf", ".png", ".svg"):
    output_path = OUTPUT_DIRECTORY / f"Figure_S24_Amph_Chemometry{suffix}"
    assert output_path.is_file() and output_path.stat().st_size > 0
with Image.open(OUTPUT_DIRECTORY / "Figure_S24_Amph_Chemometry.png") as image:
    assert image.size == (round(FIGURE_WIDTH_IN * EXPORT_DPI), round(FIGURE_HEIGHT_IN * EXPORT_DPI))
svg_text = (OUTPUT_DIRECTORY / "Figure_S24_Amph_Chemometry.svg").read_text(encoding="utf-8")
assert "<text" in svg_text and "<image" not in svg_text.casefold()
assert (OUTPUT_DIRECTORY / "Figure_S24_Amph_Chemometry.pdf").read_bytes().count(b"/Subtype /Type3") == 0
print("Figure S24 exported: PDF, PNG, and SVG.")


Figure S24 exported: PDF, PNG, and SVG.
